# Week 1 Day 2 — UnderCover: + Missing-Document Requests
## A simple extension of the claims agent

This contribution extends Ed Donner's claims agent with a visible document-checking and ticket workflow.

**One agent, one pass through the original claims.** The agent checks required documents, creates a request when evidence is missing, and records its decision. We then inspect the results, document checks and tickets.

### Workflow
Read claim and policy → check required documents → request a missing document when needed → record decision.
 Tickets are simulated records; no messages are sent to claimants.

## 1. Setup
Import the course dependencies and locate the repository's `data` folder. This cell makes no API calls. If the repository is elsewhere, set `PROJECT_ROOT_OVERRIDE`.

In [ ]:
import json
import math
import os
import time
from pathlib import Path
from typing import Literal
from datetime import datetime, timezone
from uuid import uuid4

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

PROJECT_ROOT_OVERRIDE = None  # Example: r"C:\projects\undercover"

required = ["claims.json", "policies.json", "policy_wording.md", "evals/decisions.json"]
candidates = ([Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()]
              if PROJECT_ROOT_OVERRIDE else [Path.cwd(), *Path.cwd().parents])
PROJECT_ROOT = next(
    (root for root in candidates if all((root / "data" / name).is_file() for name in required)),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Course data folder not found. Place the notebook inside the repository "
        "or set PROJECT_ROOT_OVERRIDE."
    )
DATA = PROJECT_ROOT / "data"
print("Project root:", PROJECT_ROOT)
print("Data:", DATA)

## 2. Load the data and API key
Load the 25 original claims, policies, policy wording and reference decisions. Reference decisions are used only to score the agent's answers, never passed to the agent. The API key is not displayed.

In [ ]:
load_dotenv(PROJECT_ROOT / ".env", override=False)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
if not openrouter_api_key:
    raise ValueError("Add OPENROUTER_API_KEY to the .env file in the project root.")
print("OpenRouter API key found")

def load(name):
    """Read one of the course JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False):
    claims = load("claims.json")
    decisions = load("evals/decisions.json")
    if tiny:
        chosen = {row["claim_id"] for row in decisions if row["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [row for row in decisions if row["tiny"]]
    return claims, {row["claim_id"]: row for row in decisions}


TINY = False
claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")
assert len({c["claim_id"] for c in claims}) == len(claims)
assert all(c["claim_id"] in labels for c in claims)
if not TINY:
    assert len(claims) == 25, "The assignment expects 25 claims."
print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")

## 3. Document requirements from the policy
The checklist below records explicit requirements from policy sections 2–7. Section 11 says that a missing required document is requested and the claim waits.

For example, trip delay requires receipts and carrier confirmation; theft requires a police report. There is no universal requirement to provide a payment receipt for every claim.

The checker reads document types and nonempty content. A matched type is only a presence check: the agent must still read the content, dates and amounts. Unknown document types are marked for review. Some event-dependent requirements need the agent to identify the event first.

In [ ]:
# Explicit document requirements from policy wording version 2026.1.
# Aliases describe document metadata, not proof that the content is sufficient.
DOCUMENT_TYPES = {
    "receipts": {"receipt", "receipts"},
    "carrier_confirmation": {"carrier_report", "carrier_confirmation", "delay_confirmation"},
    "police_report": {"police_report"},
    "repair_evidence": {"repair_estimate", "repair_quote", "irreparability_statement", "repair_report"},
    "itemized_bill": {"itemized_bill", "medical_bill", "invoice"},
    "medical_records": {"medical_records", "medical_record", "medical_report"},
    "physician_confirmation": {"physician_letter", "doctor_letter", "medical_certificate", "medical_report"},
}
REQUIREMENTS = {
    "receipts": {"document_name": "Receipts for the claimed expenses"},
    "carrier_confirmation": {"document_name": "Carrier confirmation/report for the incident"},
    "police_report": {"document_name": "Police report for the theft"},
    "repair_evidence": {"document_name": "Repair estimate or statement that the item cannot be repaired"},
    "itemized_bill": {"document_name": "Itemized medical bill"},
    "medical_records": {"document_name": "Medical records from the visit"},
    "physician_confirmation": {"document_name": "Written confirmation from a physician"},
}


def document_requirements(claim_type, event):
    if claim_type == "trip_delay":
        return "3", ["receipts", "carrier_confirmation"], "Check delay duration and cause in the carrier confirmation."
    if claim_type == "baggage_delay":
        return "4", ["receipts", "carrier_confirmation"], "Check that the carrier report concerns delayed baggage."
    if claim_type == "passport":
        return "6", ["receipts"], "Also review the claimant's short account of what happened."
    if claim_type == "emergency_medical":
        return "7", ["itemized_bill", "medical_records"], "Check itemization and medical content."
    if claim_type == "baggage":
        if event == "theft":
            return "5", ["police_report"], "Check that the theft was reported within 24 hours."
        if event == "carrier_loss_or_damage":
            return "5", ["carrier_confirmation"], "Check that the report concerns this loss or damage."
        if event == "accidental_damage":
            return "5", ["repair_evidence"], "Check that the estimate or irreparability statement concerns the claimed item."
        return "5", [], "The event needs review; no universal receipt requirement is stated for baggage claims."
    if claim_type in {"trip_cancellation", "trip_interruption"}:
        if event == "illness_injury_death":
            return "2", ["physician_confirmation"], "Review the physician confirmation and eligibility."
        return "2", [], "Other grounds need contextual review; do not invent a universal receipt requirement."
    return "11", [], "Unsupported claim type: review the relevant policy section manually."


def inspect_required_documents(claim, event="unspecified"):
    section, required, note = document_requirements(claim["claim_type"], event)
    documents = claim.get("documents", [])
    known_types = set().union(*DOCUMENT_TYPES.values())
    unclassified = [d.get("name", "Unnamed document") for d in documents
                    if str(d.get("content", "")).strip()
                    and str(d.get("type", "")).strip().casefold() not in known_types]
    checks = []
    for requirement_id in required:
        matches = [d.get("name", "Unnamed document") for d in documents
                   if str(d.get("content", "")).strip()
                   and str(d.get("type", "")).strip().casefold() in DOCUMENT_TYPES[requirement_id]]
        # Unknown metadata may hide valid evidence; do not call it definitely missing.
        status = "present_by_type" if matches else ("needs_review" if unclassified else "missing")
        checks.append({"requirement_id": requirement_id, **REQUIREMENTS[requirement_id],
                       "policy_section": section, "status": status, "matched_documents": matches})
    return {"claim_id": claim["claim_id"], "claim_type": claim["claim_type"],
            "event": event, "policy_section": section, "checks": checks,
            "missing": [r for r in checks if r["status"] == "missing"],
            "unclassified_documents": unclassified, "content_review_note": note,
            "scope": "Metadata completeness only; review content, expense coverage and eligibility separately."}

## 4. Agent tools
The original tools retrieve the policy and record a decision. Two tools extend the workflow:

- `check_required_documents`: returns a report showing required documents, matches and missing items.
- `create_document_request`: creates a simulated ticket only for an item marked missing by the latest check.

Checks and tickets are saved with the claim result. Duplicate requests for the same requirement are avoided. The agent must call the checker before recording its decision.

In [ ]:
def make_tools():
    state = {"claim_id": None, "policy_id": None, "claim": None,
             "recorded": None, "tickets": {}, "document_checks": []}

    @tool
    def get_policy(policy_id: str) -> str:
        """Look up the current customer's policy record."""
        if policy_id != state["policy_id"]:
            return "Error: use the current claim's policy_id."
        policy = policies.get(policy_id)
        return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"

    @tool
    def check_required_documents(
        claim_id: str,
        event: Literal["unspecified", "theft", "carrier_loss_or_damage",
                       "accidental_damage", "illness_injury_death", "other"] = "unspecified",
    ) -> str:
        """Check the actual current claim documents against explicit policy requirements.

        Select the event from the claim's facts, especially for baggage/cancellation.
        This checks metadata completeness; inspect document content separately.
        """
        if claim_id != state["claim_id"]:
            return "Error: use the current claim_id."
        report = inspect_required_documents(state["claim"], event)
        state["document_checks"].append(report)
        return json.dumps(report, ensure_ascii=False)

    @tool
    def create_document_request(claim_id: str, requirement_id: str) -> str:
        """Create a simulated ticket for a requirement marked missing in the latest check.

        Call check_required_documents first. Pass its exact missing requirement_id.
        No message is sent to the claimant.
        """
        if claim_id != state["claim_id"]:
            return "Error: use the current claim_id."
        if not state["document_checks"]:
            return "Error: check_required_documents must be called first."
        requirement = next((r for r in state["document_checks"][-1]["missing"]
                            if r["requirement_id"] == requirement_id), None)
        if requirement is None:
            return "Error: this requirement was not identified as missing in the latest check."
        if requirement_id not in state["tickets"]:
            section = requirement["policy_section"]
            name = requirement["document_name"]
            state["tickets"][requirement_id] = {
                "ticket_id": f"{claim_id}-DOC-{len(state['tickets']) + 1:02d}",
                "claim_id": claim_id, "requirement_id": requirement_id,
                "document_name": name, "policy_section": section,
                "reason": f"Required by policy section {section}; missing from submitted documents.",
                "message": f"Please provide: {name}. Required under policy section {section}.",
                "status": "open",
            }
        return json.dumps(state["tickets"][requirement_id], ensure_ascii=False)

    @tool
    def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"],
                        payout: float, rationale: str) -> str:
        """Record the current claim's decision, USD payout and a 2-3 sentence explanation."""
        if claim_id != state["claim_id"]:
            return "Error: use the current claim_id."
        if not state["document_checks"]:
            return "Error: call check_required_documents before recording the decision."
        if not math.isfinite(payout) or payout < 0 or (decision != "approve" and payout != 0):
            return "Error: use a finite nonnegative payout; decline/refer require zero."
        state["recorded"] = dict(decision=decision, payout=payout, rationale=rationale)
        return f"Recorded {decision} on {claim_id}"

    tools = [get_policy, check_required_documents, create_document_request, record_decision]
    return tools, state

## 5. Create the agent
Use the model setting from the original lab. Change `MODEL` to try another tool-capable model; set `EFFORT = None` if it does not support reasoning effort. The identifier below is inherited from the source notebook, not a claim about current availability.

The instructions tell the agent to inspect the documents, create a ticket for a required missing document when it prevents a decision, and record `refer` with payout 0. A clear exclusion can still justify declining without requesting irrelevant documents.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""



def is_right(answer, label):
    """Same scoring rule as the original lab."""
    return (answer["decision"] == label["decision"]
            and abs(answer["payout"] - label["payout"]) <= 1)


def make_claim_prompt(claim):
    """The claim includes submitted documents; the policy comes from a tool."""
    return "# Claim\n\n" + json.dumps(claim, indent=2)

DOCUMENT_WORKFLOW = """Use get_policy to retrieve the customer's policy record.
The claim contains the submitted documents in its documents field.
1. Read the claim, policy wording and document contents.
2. Call check_required_documents before recording any decision. Classify the event
   from the facts; do not infer it from the desired decision.
3. Inspect matched documents for relevant content, dates, amounts and support for
   each claimed expense. The checklist is only a metadata completeness check.
4. Review unclassified documents yourself. Do not assume needs_review means missing.
5. If the claim is clearly excluded, decline without requesting documents that
   would not change the outcome.
6. Otherwise, if a required document is marked missing and prevents a decision,
   call create_document_request for each exact missing requirement_id. Then refer
   with payout=0, as the claim waits under section 11.
7. If content is inadequate, facts conflict, or wording does not settle the question,
   refer with payout=0 and explain the issue. Do not invent requirements or force
   a ticket for an item the checker has not identified as missing.
8. Otherwise decide the claim and calculate payout using section 10.
A ticket is not a sent message. Do not claim the claimant has been contacted.
"""

MODEL = "openai/gpt-6-luna"
EFFORT = "low"

settings = dict(model=MODEL, base_url=OPENROUTER_BASE_URL,
                api_key=openrouter_api_key, timeout=120, max_retries=2)
if EFFORT is not None:
    settings["reasoning_effort"] = EFFORT

llm = ChatOpenAI(**settings)
tools, state = make_tools()
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=f"{INSTRUCTIONS}\n{DOCUMENT_WORKFLOW}\n\n{policy_wording}",
)

## 6. Process one claim
Clear the previous claim's state, run the agent, and collect its decision, document checks and tickets. Score the decision using the original lab's rule: the same decision and a payout within $1 of the reference.

Record elapsed time and the sum of costs reported in AI-message metadata. Missing cost information or a failed invocation is shown as unknown, not as zero. A failed invocation or a missing decision counts as incorrect.

In [ ]:
def message_cost(message):
    usage = (getattr(message, "response_metadata", None) or {}).get("token_usage") or {}
    value = usage.get("cost")
    if value is None:
        return None
    value = float(value)
    return value if math.isfinite(value) and value >= 0 else None


def decide_with_agent(claim):
    state.update(claim_id=claim["claim_id"], policy_id=claim["policy_id"],
                 claim=claim, recorded=None, tickets={}, document_checks=[])
    start = time.perf_counter()
    result, error = None, ""
    try:
        result = agent.invoke(
            {"messages": [{"role": "user", "content": make_claim_prompt(claim)}]},
            config={"recursion_limit": 30},
        )
    except Exception as exc:
        # Do not save exception text: it may contain HTTP request details.
        error = type(exc).__name__

    messages = result["messages"] if result else []
    ai_messages = [m for m in messages if m.type == "ai"]
    costs = [message_cost(m) for m in ai_messages]
    cost_known = not error and bool(costs) and all(c is not None for c in costs)
    answer = state["recorded"] or {
        "decision": "nothing recorded", "payout": 0.0, "rationale": "",
    }
    label = labels[claim["claim_id"]]
    return {
        "model": MODEL, "effort": EFFORT or "default",
        "claim_id": claim["claim_id"], **answer,
        "difficulty": label["difficulty"],
        "right": not bool(error) and is_right(answer, label),
        "cost": sum(costs) if cost_known else None,
        "seconds": round(time.perf_counter() - start, 3),
        "ai_calls": len(ai_messages) if result else None,
        "tickets": list(state["tickets"].values()),
        "document_checks": list(state["document_checks"]), "error": error,
    }

## 7. Run the agent once on the claims
**This cell makes paid API calls.** Each claim may need several model calls. It runs one agent through the queue once, without a baseline comparison or repeated experiments.

Results are saved after every completed claim to a new JSON file. A restart does not require another paid run just to view completed results. An interrupted claim is not automatically resumed.

In [ ]:
results_folder = PROJECT_ROOT / "community_contributions" / "oleg" / "document_request_results"
results_folder.mkdir(parents=True, exist_ok=True)
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
results_path = results_folder / f"{run_id}.json"

payload = {"model": MODEL, "effort": EFFORT, "claim_count": len(claims), "results": []}
for claim in claims:
    row = decide_with_agent(claim)
    payload["results"].append(row)
    temporary = results_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False),
                         encoding="utf-8")
    temporary.replace(results_path)
    print(f"{row['claim_id']} | {row['decision']} | right={row['right']} | "
          f"checks={len(row['document_checks'])} | tickets={len(row['tickets'])} | "
          f"error={row['error'] or '-'}")
print("Saved to:", results_path)

### Reopen saved results without calling the model
Leave `LOAD_PATH = None` after a fresh run. After a restart, run setup and data loading, enter the saved JSON path here, and continue with the tables below. Skip agent creation and the paid run cell.

In [ ]:
LOAD_PATH = None  # Example: r"C:\projects\undercover\...\20261022T120000Z_ab12cd34.json"
if LOAD_PATH:
    payload = json.loads(Path(LOAD_PATH).read_text(encoding="utf-8"))
    print("Loaded:", LOAD_PATH)

## 8. Results at a glance
One row per processed claim: decision, payout, correctness, checks and ticket count. The summary shows processing cost and average time. This single run does not establish consistent performance or identify the cheapest model.

In [ ]:
rows = payload["results"]
results_table = pd.DataFrame([
    {"claim_id": r["claim_id"], "decision": r["decision"], "payout": r["payout"],
     "right": r["right"], "checks": len(r["document_checks"]), "tickets": len(r["tickets"]),
     "cost_usd": r["cost"], "seconds": r["seconds"], "error": r["error"]}
    for r in rows
])
right = sum(r["right"] for r in rows)
known_costs = [r["cost"] for r in rows if r["cost"] is not None]
print(f"Processed: {len(rows)} / {payload['claim_count']} | Correct: {right} / {len(rows)}")
print("Recorded cost:", f"${sum(known_costs):.5f}" if rows and len(known_costs) == len(rows)
      else "Unknown: some claim costs are missing.")
if rows:
    print(f"Average time: {sum(r['seconds'] for r in rows) / len(rows):.2f} seconds per claim")
display(results_table)

## 9. Document checks and tickets
The first table shows each checked requirement, matching documents and its status. `present_by_type` means a matching nonempty document was found; `needs_review` is not proof of absence. The second table shows actual requests created by the agent.

An empty ticket table is a valid result when no request was created. These are saved ticket records, not messages sent to claimants.

In [ ]:
check_rows = []
ticket_rows = []
for row in rows:
    for report in row["document_checks"]:
        for check in report["checks"]:
            check_rows.append({"claim_id": row["claim_id"],
                               "section": check["policy_section"],
                               "required_document": check["document_name"],
                               "status": check["status"],
                               "matched_documents": "; ".join(check["matched_documents"])})
    ticket_rows.extend(row["tickets"])
print("Document checks")
display(pd.DataFrame(check_rows, columns=[
    "claim_id", "section", "required_document", "status", "matched_documents",
]))
print(f"Tickets created: {len(ticket_rows)}")
display(pd.DataFrame(ticket_rows, columns=[
    "ticket_id", "claim_id", "document_name", "policy_section", "message", "status",
]))

## 10. Inspect one claim
Change `CLAIM_ID` to inspect any processed claim. By default, show the first claim with a ticket, or the first processed claim if no tickets exist.

View the submitted document contents, the checker's complete report, the request text and the final decision together. This cell makes no API calls. A report with an empty checklist still requires contextual review; it does not establish that all evidence is sufficient.

In [ ]:
CLAIM_ID = None  # Example: "CLM-1017"

if rows:
    selected_id = CLAIM_ID or next((r["claim_id"] for r in rows if r["tickets"]), rows[0]["claim_id"])
    selected = next((r for r in rows if r["claim_id"] == selected_id), None)
    original = next((c for c in claims if c["claim_id"] == selected_id), None)
    if selected is None or original is None:
        print("Claim not found. Choose a processed claim_id from the results table.")
    else:
        print("CLAIM:", selected_id)
        print(original.get("summary", ""))
        print("SUBMITTED DOCUMENTS")
        display(pd.DataFrame(original.get("documents", [])))
        print("DOCUMENT CHECK REPORTS")
        for report in selected["document_checks"]:
            display(report)
        print("CREATED REQUESTS")
        for ticket in selected["tickets"]:
            print(ticket["ticket_id"], "| Policy section", ticket["policy_section"])
            print(ticket["message"])
        if not selected["tickets"]:
            print("No request was created.")
        print("DECISION:", selected["decision"], "| Payout:", selected["payout"])
        print(selected["rationale"])
        print("Matches the reference:", selected["right"])
else:
    print("No processed claims to inspect yet.")

## Performance Summary

This table summarizes the agent's results:

- **Accuracy:** correct decisions and payouts compared with the reference answers.
- **Total cost:** the sum of reported API costs for processed claims.
- **Total time:** elapsed processing time across all claims.
- **Speed:** average processing time per claim.
- **Tickets:** the number of document requests created.

The table uses saved results and makes no additional API calls. Missing cost information is shown as **Unknown**.

In [ ]:
rows = payload["results"]

count = len(rows)
right = sum(row["right"] for row in rows)
seconds = sum(row["seconds"] for row in rows)

costs_known = count > 0 and all(
    row["cost"] is not None for row in rows
)
cost = sum(row["cost"] for row in rows) if costs_known else None

summary = pd.DataFrame([{
    "Model": payload["model"],
    "Reasoning": payload["effort"] or "default",
    "Processed": f"{count} / {payload['claim_count']}",
    "Correct": f"{right} / {count}",
    "Accuracy": f"{right / count:.1%}" if count else "—",
    "Total cost": f"${cost:.5f}" if cost is not None else "Unknown",
    "Total time": f"{seconds:.1f} seconds",
    "Speed": f"{seconds / count:.2f} seconds per claim" if count else "—",
    "Tickets": sum(len(row["tickets"]) for row in rows),
}])

display(summary)

## What this extension adds

The agent now produces a visible document-check report and can create a policy-based request ticket when a required document is missing. This makes the next action explicit instead of only recording `refer`.

The notebook processes the original claims once and displays decisions, checks and tickets. Results are saved for later inspection. It does not send requests or receive new documents from claimants.

Document-type checks are deliberately limited: the model still reviews the content and coverage. This extension is a workflow demonstration; any accuracy, cost or consistency claim must come from measured runs.